In [42]:
%%writefile phase1_identity.py
import os
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

# ===================== CONFIG =====================

VOTER_CSV = "data/voter_details.csv"

REQUIRED_COLUMNS = {
    "voter_id",
    "aadhaar_id",
    "name",
    "village",
    "voting_center",
    "face_id",
    "has_voted"
}

# ===================== LOAD DATA =====================
def load_voters():

    if not os.path.exists(VOTER_CSV):
        raise FileNotFoundError("voter_details.csv not found in data folder")

    df = pd.read_csv(VOTER_CSV)

    if not REQUIRED_COLUMNS.issubset(df.columns):
        raise ValueError("CSV columns do not match required format")

    return df


# ===================== SEARCH VOTER =====================
def get_voter(voter_input):

    voter_input = str(voter_input).strip()
    df = load_voters()

    match = df[
        (df["voter_id"].astype(str) == voter_input) |
        (df["aadhaar_id"].astype(str) == voter_input)
    ]

    if match.empty:
        return None

    voter = match.iloc[0]

    # 🔒 Prevent double voting (Yes / No format)
    if str(voter["has_voted"]).strip().lower() == "yes":
        return "ALREADY_VOTED"

    return {
        "voter_id": voter["voter_id"],
        "face_id": voter["face_id"],
        "name": voter["name"],
        "village": voter["village"],
        "voting_center": voter["voting_center"]
    }


# ===================== VERIFY IDENTITY (FOR FLASK) =====================
def verify_identity(voter_input):

    try:
        result = get_voter(voter_input)

        if result is None:
            return {
                "status": False,
                "reason": "Invalid Voter ID / Aadhaar ID"
            }

        if result == "ALREADY_VOTED":
            return {
                "status": False,
                "reason": "You have already voted."
            }

        return {
            "status": True,
            "data": result
        }

    except Exception as e:
        print("ERROR:", e)   # 👈 ADD THIS
        return {
            "status": False,
            "reason": str(e)  # 👈 SHOW REAL ERROR
        }


Overwriting phase1_identity.py


In [43]:
%%writefile phase2_face_liveness.py
import cv2
import os
import base64
import numpy as np
import face_recognition
import random

# ================= CONFIG =================

DATASET_DIR = "static/images/voters/"
MATCH_TOLERANCE = 0.45
NOSE_MOVEMENT_THRESHOLD = 12
BLINK_THRESHOLD = 4
MAX_ATTEMPTS = 3   # 🔒 Maximum allowed failures

# ================= GLOBAL STATE =================

liveness_state = {}

# ================= RESET =================

def reset_liveness():
    global liveness_state

    challenges = [
        "BLINK",
        "TURN_LEFT",
        "TURN_RIGHT",
        "LOOK_UP",
        "LOOK_DOWN"
    ]

    liveness_state = {
        "face_matched": False,
        "challenge": random.choice(challenges),
        "challenge_done": False,
        "neutral_x": None,
        "neutral_y": None,
        "known_encoding": None,
        "attempt_count": 0   # 🔥 Track failures
    }

# ================= LOAD REGISTERED FACE =================

def load_registered_face(face_id):

    folder = os.path.join(DATASET_DIR, face_id)
    if not os.path.exists(folder):
        return None

    for file in os.listdir(folder):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(folder, file)
            image = face_recognition.load_image_file(path)
            encodings = face_recognition.face_encodings(image)
            if encodings:
                return encodings[0]

    return None

# ================= DECODE =================

def decode_base64_image(frame_data):
    header, encoded = frame_data.split(",", 1)
    img_bytes = base64.b64decode(encoded)
    np_arr = np.frombuffer(img_bytes, np.uint8)
    return cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

# ================= MAIN =================

def process_frame(face_id, frame_data):

    global liveness_state

    if not liveness_state:
        reset_liveness()

    # 🔒 Check attempt limit
    if liveness_state["attempt_count"] >= MAX_ATTEMPTS:
        return {
            "status": False,
            "locked": True,
            "message": "Verification blocked. Maximum attempts exceeded."
        }

    try:
        frame = decode_base64_image(frame_data)
    except:
        return {"status": False, "message": "Invalid image data"}

    frame = cv2.resize(frame, (320, 240))
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # ================= STEP 1: FACE MATCH =================

    if not liveness_state["face_matched"]:

        if liveness_state["known_encoding"] is None:
            known = load_registered_face(face_id)
            if known is None:
                return {"status": False, "message": "No registered face found"}
            liveness_state["known_encoding"] = known

        known_encoding = liveness_state["known_encoding"]

        locations = face_recognition.face_locations(rgb, model="hog")

        if len(locations) != 1:
            return {"status": False, "message": "Show exactly one face"}

        encodings = face_recognition.face_encodings(rgb, locations)

        if not encodings:
            return {"status": False, "message": "Face encoding failed"}

        match = face_recognition.compare_faces(
            [known_encoding],
            encodings[0],
            tolerance=MATCH_TOLERANCE
        )

        if not match[0]:
            liveness_state["attempt_count"] += 1   # 🔥 Increase failure count
            remaining = MAX_ATTEMPTS - liveness_state["attempt_count"]

            return {
                "status": False,
                "message": f"Face mismatch. Attempts left: {remaining}"
            }

        liveness_state["face_matched"] = True

        return {
            "status": False,
            "message": f"Face matched. Please {liveness_state['challenge']}"
        }

    # ================= STEP 2: LIVENESS =================

    landmarks_list = face_recognition.face_landmarks(rgb)

    if not landmarks_list:
        return {"status": False, "message": "Face not detected"}

    landmarks = landmarks_list[0]
    challenge = liveness_state["challenge"]

    nose = landmarks["nose_tip"]
    nose_x = np.mean([p[0] for p in nose])
    nose_y = np.mean([p[1] for p in nose])

    if liveness_state["neutral_x"] is None:
        liveness_state["neutral_x"] = nose_x
        liveness_state["neutral_y"] = nose_y
        return {"status": False, "message": f"Please {challenge}"}

    nx = liveness_state["neutral_x"]
    ny = liveness_state["neutral_y"]

    if challenge == "LOOK_UP" and nose_y < ny - NOSE_MOVEMENT_THRESHOLD:
        liveness_state["challenge_done"] = True

    elif challenge == "LOOK_DOWN" and nose_y > ny + NOSE_MOVEMENT_THRESHOLD:
        liveness_state["challenge_done"] = True

    elif challenge == "TURN_LEFT" and nose_x < nx - NOSE_MOVEMENT_THRESHOLD:
        liveness_state["challenge_done"] = True

    elif challenge == "TURN_RIGHT" and nose_x > nx + NOSE_MOVEMENT_THRESHOLD:
        liveness_state["challenge_done"] = True

    elif challenge == "BLINK":
        left_eye = landmarks["left_eye"]
        right_eye = landmarks["right_eye"]

        left_height = abs(left_eye[1][1] - left_eye[5][1])
        right_height = abs(right_eye[1][1] - right_eye[5][1])

        if left_height < BLINK_THRESHOLD and right_height < BLINK_THRESHOLD:
            liveness_state["challenge_done"] = True

    # ================= FINAL =================

    if liveness_state["challenge_done"]:
        reset_liveness()
        return {"status": True}

    return {
        "status": False,
        "message": f"Please {challenge}"
    }

Overwriting phase2_face_liveness.py


In [44]:
%%writefile phase3_offline_otp.py
import random
import time
import pandas as pd
import os

OTP_FILE = "data/otp_sessions.csv"
OTP_VALIDITY = 120  # seconds
AUTO_FILE = "data/auto_mode.txt"

os.makedirs("data", exist_ok=True)


# =====================================================
# GENERATE OTP
# =====================================================

def generate_otp():
    return str(random.randint(100000, 999999))


# =====================================================
# START OTP SESSION
# =====================================================

def start_otp_session(session, voter_id):

    otp = generate_otp()

    auto_mode = get_auto_mode()

    status = "APPROVED" if auto_mode == "ON" else "WAITING"

    data = {
        "voter_id": voter_id,
        "otp": otp,
        "status": status,
        "timestamp": time.time()
    }

    df = pd.DataFrame([data])

    if os.path.exists(OTP_FILE):
        old = pd.read_csv(OTP_FILE)
        old = old[old["voter_id"] != voter_id]
        df = pd.concat([old, df])

    df.to_csv(OTP_FILE, index=False)

    session["otp_session_started"] = True
    return True


# =====================================================
# APPROVE OTP
# =====================================================

def approve_otp(voter_id):

    if not os.path.exists(OTP_FILE):
        return False

    df = pd.read_csv(OTP_FILE)

    df.loc[df["voter_id"] == voter_id, "status"] = "APPROVED"

    df.to_csv(OTP_FILE, index=False)

    return True


# =====================================================
# REJECT OTP
# =====================================================

def reject_otp(voter_id):

    if not os.path.exists(OTP_FILE):
        return False

    df = pd.read_csv(OTP_FILE)

    df.loc[df["voter_id"] == voter_id, "status"] = "REJECTED"

    df.to_csv(OTP_FILE, index=False)

    return True


# =====================================================
# AUTO MODE CONTROL
# =====================================================

def set_auto_mode(status):
    with open(AUTO_FILE, "w") as f:
        f.write(status)


def get_auto_mode():
    if not os.path.exists(AUTO_FILE):
        return "OFF"
    with open(AUTO_FILE, "r") as f:
        return f.read().strip()


# =====================================================
# CHECK OTP STATUS (WITH EXPIRY)
# =====================================================

def check_otp_status(voter_id):

    if not os.path.exists(OTP_FILE):
        return {"status": "NO_SESSION"}

    df = pd.read_csv(OTP_FILE)
    row = df[df["voter_id"] == voter_id]

    if row.empty:
        return {"status": "NO_SESSION"}

    record = row.iloc[0]

    # Expiry Check
    if record["status"] == "WAITING":
        current_time = time.time()
        if current_time - float(record["timestamp"]) > OTP_VALIDITY:
            df.loc[df["voter_id"] == voter_id, "status"] = "EXPIRED"
            df.to_csv(OTP_FILE, index=False)
            return {"status": "EXPIRED"}

    return {"status": record["status"]}

Overwriting phase3_offline_otp.py


In [45]:
%%writefile phase4_voting.py
# phase4_voting.py

import pandas as pd
import os

# ===================== PATHS =====================

CANDIDATE_CSV = os.path.join("data", "candidates.csv")
VOTER_CSV = os.path.join("data", "voter_details.csv")


# ===================== LOAD CANDIDATES =====================
def load_candidates():
    """
    Returns list of candidates for rendering in voting.html
    """
    if not os.path.exists(CANDIDATE_CSV):
        return []

    df = pd.read_csv(CANDIDATE_CSV)

    # Ensure votes column exists
    if "votes" not in df.columns:
        df["votes"] = 0

    return df.to_dict(orient="records")


# ===================== CHECK IF VOTER EXISTS =====================
def voter_exists(voter_id):
    if not os.path.exists(VOTER_CSV):
        return False

    df = pd.read_csv(VOTER_CSV)

    return not df[df["voter_id"].astype(str) == str(voter_id)].empty


# ===================== CHECK DOUBLE VOTING =====================
def has_already_voted(voter_id):
    """
    Returns True if voter has already voted.
    Blocks unknown voters for safety.
    """
    if not os.path.exists(VOTER_CSV):
        return True  # Safety block

    df = pd.read_csv(VOTER_CSV)

    row = df[df["voter_id"].astype(str) == str(voter_id)]


    if row.empty:
        return True  # Unknown voter blocked

    status = str(row.iloc[0]["has_voted"]).strip().lower()

    return status == "yes"


# ===================== INCREMENT CANDIDATE VOTE =====================
def increment_candidate_vote(candidate_id):

    if not os.path.exists(CANDIDATE_CSV):
        return False

    df = pd.read_csv(CANDIDATE_CSV)
    df["candidate_id"] = df["candidate_id"].astype(str)
    if str(candidate_id) not in df["candidate_id"].values:

        return False

    df.loc[df["candidate_id"] == candidate_id, "votes"] += 1

    df.to_csv(CANDIDATE_CSV, index=False)

    return True


# ===================== MARK VOTER AS VOTED =====================
def mark_voter_voted(voter_id):

    df = pd.read_csv(VOTER_CSV)

    
    df.loc[df["voter_id"].astype(str) == str(voter_id), "has_voted"] = "Yes"


    df.to_csv(VOTER_CSV, index=False)


# ===================== CAST VOTE =====================
def cast_vote(voter_id, candidate_id):
    """
    Main function called from Flask after form submission.
    Handles:
    - Double vote prevention
    - Candidate validation
    - Safe vote update
    """

    # 🚫 1. Check voter exists
    if not voter_exists(voter_id):
        return {"status": False, "reason": "INVALID_VOTER"}

    # 🚫 2. Double voting protection
    if has_already_voted(voter_id):
        return {"status": False, "reason": "ALREADY_VOTED"}

    # 🚫 3. Validate candidate
    if not os.path.exists(CANDIDATE_CSV):
        return {"status": False, "reason": "SYSTEM_ERROR"}

    candidates = pd.read_csv(CANDIDATE_CSV)
    candidates["candidate_id"] = candidates["candidate_id"].astype(str)
    selected = candidates[candidates["candidate_id"] == str(candidate_id)]


    

    if selected.empty:
        return {"status": False, "reason": "INVALID_CANDIDATE"}

    # ✅ 4. Secure vote update
    success = increment_candidate_vote(candidate_id)

    if not success:
        return {"status": False, "reason": "SYSTEM_ERROR"}

    # ✅ 5. Mark voter
    mark_voter_voted(voter_id)

    return {"status": True}


Overwriting phase4_voting.py


In [46]:
%%writefile phase5_results.py
# phase5_results.py

import os
import hashlib
import pandas as pd

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch


# ===================== FILE PATHS =====================

CANDIDATE_FILE = os.path.join("data", "candidates.csv")
RESULT_DIR = "results"
CSV_FILE = os.path.join(RESULT_DIR, "results.csv")
PDF_FILE = os.path.join(RESULT_DIR, "results.pdf")


# ===================== ADMIN CREDENTIALS =====================

ADMIN_USER = "admin"
ADMIN_PASS_HASH = hashlib.sha256("admin123".encode()).hexdigest()


# ===================== ADMIN AUTH =====================

def verify_admin(username, password):
    """
    Verifies admin credentials using SHA256 hash.
    """
    password_hash = hashlib.sha256(password.encode()).hexdigest()
    return username == ADMIN_USER and password_hash == ADMIN_PASS_HASH



# ===================== LOAD & PROCESS RESULTS =====================

def get_results_data():
    """
    Returns:
    - Sorted result list
    - Winner(s)
    - Tie status
    """

    if not os.path.exists(CANDIDATE_FILE):
        return None

    df = pd.read_csv(CANDIDATE_FILE)

    if df.empty:
        return None

    # Ensure votes column exists
    if "votes" not in df.columns:
        df["votes"] = 0

    # Sort descending by votes
    df = df.sort_values(by="votes", ascending=False)

    max_votes = df.iloc[0]["votes"]
    winners = df[df["votes"] == max_votes]

    return {
        "data": df.to_dict(orient="records"),
        "winners": winners.to_dict(orient="records"),
        "tie": len(winners) > 1
    }


# ===================== EXPORT CSV =====================

def save_csv(df):
    """
    Saves results as CSV file.
    """
    os.makedirs(RESULT_DIR, exist_ok=True)
    df.to_csv(CSV_FILE, index=False)


# ===================== EXPORT PDF (Professional Layout) =====================

def save_pdf(df):
    """
    Generates professional PDF using ReportLab Platypus.
    """
    os.makedirs(RESULT_DIR, exist_ok=True)

    doc = SimpleDocTemplate(PDF_FILE, pagesize=A4)
    elements = []

    styles = getSampleStyleSheet()
    title_style = styles["Heading1"]
    normal_style = styles["Normal"]

    elements.append(Paragraph("Election Results Summary", title_style))
    elements.append(Spacer(1, 0.4 * inch))

    for _, row in df.iterrows():
        
        candidate = row.get("leader_name", "Unknown")

        party = row.get("party_name", "Independent")
        votes = int(row.get("votes", 0))

        text = f"{candidate} ({party}) - {votes} votes"
        elements.append(Paragraph(text, normal_style))
        elements.append(Spacer(1, 0.25 * inch))

    doc.build(elements)


# ===================== GENERATE ALL RESULT FILES =====================

def generate_result_files():
    """
    Creates both CSV and PDF result files.
    Returns True if successful.
    """

    if not os.path.exists(CANDIDATE_FILE):
        return False

    df = pd.read_csv(CANDIDATE_FILE)

    if df.empty:
        return False

    # Ensure votes column exists
    if "votes" not in df.columns:
        df["votes"] = 0

    df = df.sort_values(by="votes", ascending=False)

    save_csv(df)
    save_pdf(df)

    return True


Overwriting phase5_results.py


In [47]:
%%writefile static/css/style.css

/* ================= GLOBAL STYLES ================= */

body {
    margin: 0;
    padding: 0;
    font-family: 'Segoe UI', Arial, sans-serif;
    background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
    color: white;
    text-align: center;
}

.container {
    margin-top: 80px;
    padding: 20px;
}

h1 {
    font-size: 42px;
    margin-bottom: 10px;
    letter-spacing: 1px;
}

h2, h3 {
    margin-bottom: 10px;
}

.subtitle {
    font-size: 18px;
    margin-bottom: 30px;
    opacity: 0.9;
}

footer {
    margin-top: 60px;
    font-size: 14px;
    opacity: 0.7;
}

/* ================= BUTTONS ================= */

button {
    padding: 12px 28px;
    font-size: 16px;
    border: none;
    border-radius: 8px;
    cursor: pointer;
    margin: 10px;
    transition: all 0.3s ease;
    font-weight: 600;
}

.primary-btn {
    background-color: #28a745;
    color: white;
}

.primary-btn:hover {
    background-color: #218838;
    transform: translateY(-2px);
}


.secondary-btn {
    background-color: #6f42c1;
    color: white;
}

.secondary-btn:hover {
    background-color: #5a32a3;
    transform: translateY(-2px);
}

/* ================= LINKS ================= */

a {
    color: #ffffff;
    text-decoration: none;
    font-weight: 500;
}

a:hover {
    text-decoration: underline;
}

.back-link {
    margin-top: 40px;
}

/* ================= FEATURES SECTION ================= */

.features ul {
    list-style: none;
    padding: 0;
    margin-bottom: 40px;
}

.features li {
    margin: 8px 0;
    font-size: 17px;
}

/* ================= FORM STYLING ================= */

.form-box {
    margin-top: 30px;
}

input[type="text"],
input[type="password"] {
    padding: 12px;
    width: 260px;
    border-radius: 6px;
    border: none;
    font-size: 16px;
    margin-bottom: 15px;
    outline: none;
}

input:focus {
    box-shadow: 0 0 5px #00c6ff;
}

.error-box {
    margin-top: 20px;
    color: #ff4d4d;
    font-weight: bold;
}

.info-box {
    margin-top: 20px;
    text-align: left;
    display: inline-block;
    background: rgba(255,255,255,0.1);
    padding: 15px 20px;
    border-radius: 8px;
}

.info-box ul {
    list-style: none;
    padding: 0;
}

.info-box li {
    margin: 8px 0;
}

/* ================= VOTING PAGE ================= */

.candidate-grid {
    display: flex;
    justify-content: center;
    flex-wrap: wrap;
    gap: 30px;
    margin-top: 30px;
}

.candidate-card {
    background: white;
    color: black;
    padding: 20px;
    border-radius: 12px;
    width: 230px;
    text-align: center;
    transition: all 0.3s ease;
    box-shadow: 0 4px 15px rgba(0,0,0,0.2);
}

.candidate-card:hover {
    transform: scale(1.05);
}

.leader-img {
    width: 120px;
    height: 120px;
    object-fit: cover;
    border-radius: 10px;
}

.symbol-img {
    width: 60px;
    margin-top: 10px;
}

.vote-select {
    margin-top: 15px;
}

/* ================= RESULTS PAGE ================= */

/* ================= RESULTS PAGE ================= */

.results-box {
    margin-top: 30px;
    background: white;
    color: #222;
    padding: 30px;
    border-radius: 15px;
    width: 85%;
    margin-left: auto;
    margin-right: auto;
    box-shadow: 0 5px 20px rgba(0,0,0,0.2);
}

.result-card {
    background: white;
    color: black;
    padding: 15px;
    margin: 10px auto;
    width: 280px;
    border-radius: 10px;
    font-weight: 600;
    box-shadow: 0 4px 10px rgba(0,0,0,0.2);
}

.winner-text {
    margin: 30px auto;
    padding: 20px;
    width: 350px;
    background: rgba(0, 255, 128, 0.15);
    border: 2px solid #00e676;
    border-radius: 12px;
    font-size: 22px;
    font-weight: bold;
    color: #ffffff;
    text-align: center;
    box-shadow: 0 0 15px rgba(0, 230, 118, 0.4);
}





/* ================= CHART ================= */

canvas {
    background: white;
    padding: 20px;
    border-radius: 12px;
    margin-top: 20px;
}

/* ================= ADMIN / DASHBOARD LOOK ================= */

.admin-box {
    background: rgba(255,255,255,0.1);
    padding: 20px;
    border-radius: 10px;
    display: inline-block;
    margin-top: 30px;
}
.party-box {
    margin-top: 8px;
    padding: 6px 10px;
    background-color: #f1f1f1;
    border: 2px solid #ccc;
    border-radius: 6px;
    font-size: 14px;
    font-weight: 600;
    text-align: center;
    color: #333;
}
.success-title {
    font-size: 30px;
    margin-bottom: 20px;
    color: #28a745;
}

/* Animated Emoji */
.animated-icon {
    display: inline-block;
    margin-right: 10px;
    font-size: 36px;
    animation: popBounce 1s ease forwards;
}

/* Pop + Bounce Effect */
@keyframes popBounce {
    0% {
        transform: scale(0.2);
        opacity: 0;
    }
    50% {
        transform: scale(1.3);
    }
    70% {
        transform: scale(0.9);
    }
    100% {
        transform: scale(1);
        opacity: 1;
    }
}


/* Align buttons in one row like screenshot */
.button-group {
    margin-top: 30px;
}

/* Officer Button Style */
.officer-btn {
    background-color: #ffc107;
    color: black;
}

.officer-btn:hover {
    background-color: #e0a800;
    transform: translateY(-2px);
}


Overwriting static/css/style.css


In [48]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Smart Voting System</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='css/style.css') }}">
</head>
<body>

    <div class="container">

        <h1>🗳 Smart Voting System</h1>

        <p class="subtitle">
            Secure Multi-Layer Authentication Based Voting Platform
        </p>

        <div class="info-box">
            <ul>
                <li>✔ Identity Verification</li>
                <li>✔ Face Liveness Detection</li>
                <li>✔ Offline OTP Authentication</li>
                <li>✔ Secure Vote Casting</li>
                <li>✔ Admin Protected Results</li>
            </ul>
        </div>

        <!-- BUTTON SECTION -->
        <div class="button-group">

            <a href="{{ url_for('identity') }}">
                <button class="primary-btn">🚀 Start Voting </button>
            </a>

            <a href="{{ url_for('officer_login') }}">
                <button class="officer-btn">👮 Officer Login</button>
            </a>

            <a href="{{ url_for('admin') }}">
                <button class="secondary-btn">🛡 Admin Login</button>
            </a>

        </div>

        <footer>
            <p>© 2026 Smart Voting System | Final Year Project</p>
        </footer>

    </div>

</body>
</html>


Overwriting templates/index.html


In [49]:
%%writefile templates/identity.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Identity Verification</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='css/style.css') }}">
</head>
<body>

    <div class="container">

        <h1>🆔 Identity Verification</h1>

        <p class="subtitle">
            Please enter your registered Voter ID to proceed
        </p>

        <div class="form-box">
            <form method="POST">

                <input type="text"
                       name="voter_id"
                       placeholder="Enter Voter ID"
                       required>

                <br><br>

                <button type="submit" class="primary-btn">
                    Verify Identity
                </button>

            </form>
        </div>

        {% if error %}
        <div class="error-box">
            {{ error }}
        </div>
        {% endif %}
        <form action="{{ url_for('index') }}">
            <button type="submit" class="secondary-btn">
                ⬅ Back to Home
            </button>
        </form>


    </div>

</body>
</html>


Overwriting templates/identity.html


In [50]:
%%writefile templates/details.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Voter Details Confirmation</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='css/style.css') }}">
</head>
<body>

<div class="container">

    <h1>🧾 Confirm Voter Details</h1>

    <!-- ===== VOTER IMAGE ===== -->
<div style="margin-top:20px; text-align:center;">
    <img src="{{ url_for('static', filename=image_path) }}"
         width="200"
         style="border-radius:12px; box-shadow:0 4px 12px rgba(0,0,0,0.4);">
</div>




    <!-- ===== VOTER DETAILS ===== -->
    <div class="info-box" style="margin-top:20px;">
        <p><b>Name:</b> {{ session['voter_name'] }}</p>
        <p><b>Voter ID:</b> {{ session['voter_id'] }}</p>
        <p><b>Village:</b> {{ session['village'] }}</p>
        <p><b>Voting Center:</b> {{ session['voting_center'] }}</p>
    </div>

    <!-- ===== CONFIRM MESSAGE ===== -->
    <p style="margin-top:20px; font-weight:bold; color:#00ffcc;">
        Please verify that the above details are correct.
    </p>

    <!-- ===== BUTTONS ===== -->
    <div style="margin-top:20px; display:flex; justify-content:center; gap:20px;">

        <!-- Back Button -->
        <form action="{{ url_for('identity') }}">
            <button type="submit" class="secondary-btn">
                ⬅ Back
            </button>
        </form>

        <!-- Proceed Button -->
        <form method="POST">
            <button type="submit" class="primary-btn">
                Proceed to Face Verification ➡
            </button>
        </form>

    </div>

</div>

</body>
</html>


Overwriting templates/details.html


In [51]:
%%writefile templates/face.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Face Verification</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='css/style.css') }}">

    <style>
        .footer-bar {
            margin-top: 40px;
            padding: 15px;
            background-color: #2c3e50;
            color: white;
            text-align: center;
            border-radius: 0 0 10px 10px;
        }
    </style>
</head>

<body>

<div class="container">

    <h1>📸 Face Verification</h1>

    <p class="subtitle">
        Secure Biometric Authentication with Liveness Detection
    </p>

    <!-- Instructions -->
    <div class="info-box" style="margin-top:20px;">
        <ul style="text-align:left;">
            <li>✔ Ensure good lighting</li>
            <li>✔ Remove glasses (if possible)</li>
            <li>✔ Look straight into the camera</li>
            <li>✔ Stay still during verification</li>
            <li>✔ Make sure only one face is visible</li>
        </ul>
    </div>

    <!-- Buttons -->
    <div style="margin-top:30px;
                display:flex;
                justify-content:center;
                gap:20px;
                flex-wrap:wrap;">

        <!-- Back -->
        <form action="{{ url_for('details') }}" method="GET">
            <button type="submit" class="secondary-btn">
                ⬅ Back
            </button>
        </form>

        <!-- Start Verification -->
        <form action="{{ url_for('face_camera') }}" method="GET">
            <button type="submit" class="primary-btn">
                Start Face Verification
            </button>
        </form>

    </div>

</div>

</body>
</html>


Overwriting templates/face.html


In [52]:
%%writefile templates/face_camera.html
<!DOCTYPE html>
<html>
<head>
    <title>Face Camera</title>
    <link rel="stylesheet"
          href="{{ url_for('static', filename='css/style.css') }}">
</head>

<body>

<div class="container">

    <h1>📸 Face Verification</h1>
    <p class="subtitle">Look at the camera and blink once</p>

    <video id="video" autoplay playsinline
           style="width:480px;
                  height:360px;
                  border-radius:10px;
                  border:3px solid #2ecc71;
                  margin-top:20px;"></video>

    <div id="statusText"
         style="margin-top:15px;
                font-weight:bold;
                color:#2ecc71;">
        Initializing camera...
    </div>

</div>

<script>
    const video = document.getElementById("video");
    const statusText = document.getElementById("statusText");

    let verifying = false;

    navigator.mediaDevices.getUserMedia({ video: true })
        .then(stream => {
            video.srcObject = stream;
            statusText.innerText = "Camera ready...";
            startVerification();
        })
        .catch(() => {
            statusText.innerText = "Camera access denied.";
        });

    async function startVerification() {

        while (true) {

            if (!video.videoWidth) {
                await new Promise(r => setTimeout(r, 200));
                continue;
            }

            if (verifying) continue;

            verifying = true;
            statusText.innerText = "Verifying...";

            const canvas = document.createElement("canvas");
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;

            const ctx = canvas.getContext("2d");
            ctx.drawImage(video, 0, 0);

            const imageData = canvas.toDataURL("image/jpeg");

            try {
                const response = await fetch("/api/verify_face", {
                    method: "POST",
                    headers: { "Content-Type": "application/json" },
                    body: JSON.stringify({ image: imageData })
                });

                const data = await response.json();

                if (data.status) {
                    statusText.innerText = "Face Verified ✅";
                    window.location.href = "/otp";
                    break;
                } else {
                    statusText.innerText = data.message;
                }

            } catch (error) {
                statusText.innerText = "Verification error.";
            }

            verifying = false;

            await new Promise(r => setTimeout(r, 500)); // small delay
        }
    }
</script>

</html>


Overwriting templates/face_camera.html


In [53]:
%%writefile templates/otp.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Officer Authorization</title>

<link rel="stylesheet"
      href="{{ url_for('static', filename='css/style.css') }}">

<style>
body {
    margin: 0;
    font-family: 'Segoe UI', sans-serif;
    background: linear-gradient(to right, #0f2027, #203a43, #2c5364);
    display: flex;
    justify-content: center;
    align-items: center;
    height: 100vh;
}

.card {
    background: rgba(255,255,255,0.95);
    padding: 40px;
    width: 420px;
    border-radius: 18px;
    text-align: center;
    box-shadow: 0 20px 50px rgba(0,0,0,0.35);
    color: black;
}

.loader {
    margin: 20px auto;
    border: 6px solid #f3f3f3;
    border-top: 6px solid #28a745;
    border-radius: 50%;
    width: 55px;
    height: 55px;
    animation: spin 1s linear infinite;
}

@keyframes spin {
    0% {transform: rotate(0deg);}
    100% {transform: rotate(360deg);}
}

.status {
    margin-top: 20px;
    font-weight: bold;
    background: #fff3cd;
    padding: 10px;
    border-radius: 8px;
    color: #856404;
}

/* ===== REJECTION MODAL ===== */

.modal {
    display: none;
    position: fixed;
    z-index: 1000;
    left: 0;
    top: 0;
    width: 100%;
    height: 100%;
    background: rgba(0,0,0,0.6);
    backdrop-filter: blur(5px);
    justify-content: center;
    align-items: center;
}

.modal-content {
    background: white;
    padding: 30px;
    width: 360px;
    border-radius: 16px;
    text-align: center;
    animation: fadeInScale 0.3s ease;
    box-shadow: 0 20px 50px rgba(0,0,0,0.3);
    color: #000;   /* 👈 Force black text */
}

.modal-content h3 {
    color: #000;
}

.modal-content p {
    color: #333;
}
.modal-icon {
    font-size: 50px;
    margin-bottom: 10px;
}

.modal-btn {
    margin-top: 20px;
    padding: 10px 20px;
    border: none;
    border-radius: 8px;
    background: #dc3545;
    color: white;
    cursor: pointer;
    font-weight: bold;
    transition: 0.3s ease;
}

.modal-btn:hover {
    background: #b02a37;
    transform: scale(1.05);
}

@keyframes fadeInScale {
    from {
        transform: scale(0.8);
        opacity: 0;
    }
    to {
        transform: scale(1);
        opacity: 1;
    }
}
</style>
</head>

<body>

<div class="card">

    <div class="loader"></div>

    <h2>🔐 Officer Authorization Required</h2>
    <p>Please wait while the officer verifies this voter.</p>

    <div class="status">
        Status: Waiting for Approval...
    </div>

</div>

<!-- Rejection Modal -->
<div id="rejectionModal" class="modal">
    <div class="modal-content">
        <div class="modal-icon">❌</div>
        <h3>Authorization Rejected</h3>
        <p>The officer has rejected this voter request.</p>
        <button class="modal-btn" onclick="goHome()">Return to Home</button>
    </div>
</div>

<script>

// ===== REAL-TIME SERVER PUSH (SSE) =====
const eventSource = new EventSource("/otp_stream");

eventSource.onmessage = function(event) {

    const data = JSON.parse(event.data);

    if(data.status === "NO_SESSION"){
        console.log("Session expired");
        return;
    }

    if(data.status === "APPROVED"){
        window.location.replace("/voting");
    }

    if(data.status === "REJECTED"){
        showRejectionModal();
    }

};

// ===== BACKUP POLLING CHECK =====
setInterval(function() {

    fetch("/check_otp_status")
        .then(response => response.json())
        .then(data => {

            if (data.status === "APPROVED") {
                window.location.replace("/voting");
            }

            if (data.status === "REJECTED") {
                showRejectionModal();
            }

        });

}, 500);


// ===== MODAL FUNCTIONS =====
function showRejectionModal(){
    document.getElementById("rejectionModal").style.display = "flex";
}

function goHome(){
    window.location.replace("/");
}

</script>

</body>
</html>

Overwriting templates/otp.html


In [54]:
%%writefile templates/voting.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Official Voting Panel</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='css/style.css') }}">

    <style>
        body {
            margin: 0;
            font-family: Arial, sans-serif;
            background: linear-gradient(to right, #0f2027, #203a43, #2c5364);
        }

        .container {
            width: 90%;
            margin: 40px auto;
            background: #ffffff;
            padding: 30px;
            border-radius: 12px;
            box-shadow: 0px 10px 30px rgba(0,0,0,0.3);
            color: #333;
            transition: 0.3s ease;
        }

        .blur {
            filter: blur(6px);
            pointer-events: none;
            user-select: none;
        }

        h1 {
            text-align: center;
            color: #0b3d91;
            margin-bottom: 5px;
        }

        .subtitle {
            text-align: center;
            margin-bottom: 25px;
            color: #555;
        }

        table {
            width: 100%;
            border-collapse: collapse;
        }

        th {
            background-color: #0b3d91;
            color: white;
            padding: 12px;
            text-align: center;
        }

        td {
            padding: 12px;
            text-align: center;
            border-bottom: 1px solid #ddd;
        }

        tr:hover {
            background-color: #f1f5ff;
        }

        .leader-img {
            width: 60px;
            height: 60px;
            border-radius: 8px;
        }

        .symbol-img {
            width: 50px;
            height: 50px;
        }

        .nota-row {
            background-color: #eef3ff;
            font-weight: bold;
        }

        .primary-btn {
            display: block;
            margin: 25px auto 0;
            padding: 12px 30px;
            background-color: #0b3d91;
            color: white;
            border: none;
            border-radius: 8px;
            font-size: 16px;
            cursor: pointer;
        }

        .primary-btn:hover {
            background-color: #072c6c;
        }

        .error-box {
            margin-top: 20px;
            text-align: center;
            color: red;
            font-weight: bold;
        }

        /* ===== MODAL STYLES ===== */
        .modal {
    display: none;
    position: fixed;
    z-index: 1000;
    inset: 0;
    background: rgba(0,0,0,0.4);   /* 🔹 Dark background */
    backdrop-filter: blur(6px);    /* 🔹 Blur effect */
    justify-content: center;
    align-items: center;
}

        
.modal-content {
    background: #ffffff;
    color: #1a1a1a;
    padding: 35px;
    border-radius: 14px;
    box-shadow: 0 15px 40px rgba(0,0,0,0.25);
    animation: fadeIn 0.25s ease;
    text-align: center;   /* optional improvement */
    width: 380px;         /* optional improvement for better alignment */
}




        .modal-buttons {
            margin-top: 20px;
            display: flex;
            justify-content: space-around;
        }

        .btn-confirm {
    background: linear-gradient(135deg, #0b3d91, #1f5bd8);
    color: #ffffff;
}

.modal-buttons button {
    padding: 8px 20px;
    border-radius: 6px;
    border: none;
    cursor: pointer;
}

        .btn-cancel {
    background: #f0f0f0;
    color: #333;
}
.modal-content {
    animation: fadeIn 0.25s ease;
}

.warning-title {
    color: #c0392b;
    font-weight: bold;
}


        @keyframes fadeIn {
            from {transform: scale(0.9); opacity: 0;}
            to {transform: scale(1); opacity: 1;}
        }
    </style>
</head>

<body>

<div class="container" id="mainContainer">

    <h1>🗳 Official Electronic Voting System</h1>
    <p class="subtitle">Please select your preferred candidate below</p>

    <form method="POST" id="voteForm">

        <table>
            <tr>
                <th>Candidate ID</th>
                <th>Candidate</th>
                <th>Party Name</th>
                <th>Symbol</th>
                <th>Vote</th>
            </tr>

            {% for candidate in candidates %}
            {% if candidate.party_name != "None Of The Above" %}
            <tr>
                <td>{{ candidate.candidate_id }}</td>
                <td>
                    <img src="{{ url_for('static',
                    filename='images/leaders/' + candidate.leader_image) }}"
                    class="leader-img"><br>
                    {{ candidate.leader_name }}
                </td>
                <td>{{ candidate.party_name }}</td>
                <td>
                    <img src="{{ url_for('static',
                    filename='images/symbols/' + candidate.symbol_image) }}"
                    class="symbol-img">
                </td>
                <td>
                    <input type="radio"
                           name="selected_candidate"
                           value="{{ candidate.candidate_id }}">
                </td>
            </tr>
            {% endif %}
            {% endfor %}

            {% for candidate in candidates %}
            {% if candidate.party_name == "None Of The Above" %}
            <tr class="nota-row">
                <td>{{ candidate.candidate_id }}</td>
                <td colspan="2">None Of The Above</td>
                <td>
                    <img src="{{ url_for('static',
                    filename='images/leaders/' + candidate.leader_image) }}"
                    class="symbol-img">
                </td>
                <td>
                    <input type="radio"
                           name="selected_candidate"
                           value="{{ candidate.candidate_id }}">
                </td>
            </tr>
            {% endif %}
            {% endfor %}

        </table>

        <button type="button" class="primary-btn" onclick="openModal()">
            Submit Vote
        </button>

    </form>

    {% if error %}
    <div class="error-box">{{ error }}</div>
    {% endif %}

</div>

<!-- ===== MODAL ===== -->
<div class="modal" id="confirmModal">
    <div class="modal-content">
        <h3>Confirm Your Vote</h3>
        <p>Are you sure you want to cast your vote?<br>This action cannot be changed.</p>

        <div class="modal-buttons">
            <button class="btn-confirm" onclick="submitVote()">Yes</button>
            <button class="btn-cancel" onclick="closeModal()">No</button>
        </div>
    </div>
</div>
<!-- ===== WARNING MODAL ===== -->
<div class="modal" id="warningModal">
    <div class="modal-content">
        <h3 style="color:#c0392b;">⚠ No Candidate Selected</h3>
        <p>Please select a candidate before submitting your vote.</p>

        <div class="modal-buttons">
            <button class="btn-confirm" onclick="closeWarning()">OK</button>
        </div>
    </div>
</div>

<script>
function openModal() {
    const selected = document.querySelector('input[name="selected_candidate"]:checked');

    if (!selected) {
        document.getElementById("warningModal").style.display = "flex";
        document.getElementById("mainContainer").classList.add("blur");
        return;
    }

    // ✅ Open confirmation modal
    document.getElementById("confirmModal").style.display = "flex";
    document.getElementById("mainContainer").classList.add("blur");
}


function closeModal() {
    document.getElementById("confirmModal").style.display = "none";
    document.getElementById("mainContainer").classList.remove("blur");
}

function submitVote() {
    document.getElementById("voteForm").submit();
}
function closeWarning() {
    document.getElementById("warningModal").style.display = "none";
    document.getElementById("mainContainer").classList.remove("blur");
}

</script>

</body>
</html>

Overwriting templates/voting.html


In [55]:
%%writefile templates/vote_success.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Vote Successful</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='css/style.css') }}">

    <style>

        /* Use same background as project */
        body {
            font-family: 'Segoe UI', Arial, sans-serif;
        }

        .success-container {
            width: 50%;
            margin: 120px auto;
            background: white;
            color: #222;
            padding: 45px;
            border-radius: 16px;
            text-align: center;
            box-shadow: 0 15px 40px rgba(0,0,0,0.35);
            animation: fadeIn 0.5s ease-in-out;
        }

        .success-container h1 {
            color: #28a745; /* match your primary green */
            margin-bottom: 15px;
            font-size: 32px;
        }

        .success-container p {
            font-size: 18px;
            color: #444;
            margin: 8px 0;
        }

        .primary-btn {
            margin-top: 30px;
        }

        /* Smooth animation */
        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(15px); }
            to { opacity: 1; transform: translateY(0); }
        }

    </style>
</head>

<body>

<div class="success-container">
    
    <h1 class="success-title"><span class="animated-icon">✨Vote Cast Successfully🎉</span></h1>
    <p>🔐Your vote has been securely recorded.</p>
   
    <p>🙏Thank you for participating in the election.</p>
   


    <a href="{{ url_for('index') }}">
        <button class="primary-btn">Back to Home</button>
    </a>
</div>

</body>
</html>


Overwriting templates/vote_success.html


In [56]:
%%writefile templates/admin_login.html
<!DOCTYPE html>
<html>
<head>
    <title>Admin Login</title>
    <link rel="stylesheet"
          href="{{ url_for('static', filename='css/style.css') }}">
</head>
<body>

<div class="container">

    <h1>🔐 Admin Login</h1>

    <form method="POST">

        <input type="text" name="username"
               placeholder="Username" required><br><br>

        <input type="password" name="password"
               placeholder="Password" required><br><br>

        <button class="primary-btn">Login</button>

    </form>

    {% if error %}
    <div class="error-box">
        {{ error }}
    </div>
    {% endif %}

</div>

</body>
</html>


Overwriting templates/admin_login.html


In [57]:
%%writefile templates/results.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta http-equiv="refresh" content="5">

    <title>Election Results</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='css/style.css') }}">
          
          

    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
</head>

<body>

<div class="container">

    <h1>📊 Election Results</h1>
    <p class="subtitle">Official Vote Count Summary</p>

    <!-- ================= BAR CHART ================= -->
    {% if results %}
    <div style="width:70%; margin:auto;">
        <canvas id="voteChart"></canvas>
    </div>
    {% else %}
        <p>No results available.</p>
    {% endif %}

    <!-- ================= WINNER ================= -->
    <div class="winner-text">
        🏆 Winner: <b>{{ winner }}</b>
    </div>

    <!-- ================= STATISTICS ================= -->
    <div class="results-box">
        <h3>📈 Statistics</h3>
        <p><b>Total Voters:</b> {{ total }}</p>
        <p><b>Votes Cast:</b> {{ cast }}</p>
        <p><b>Turnout:</b> {{ turnout }}%</p>
    </div>

    <br>

    <a href="{{ url_for('index') }}">
        <button class="primary-btn">Back to Home</button>
    </a>

</div>

<!-- ================= CHART SCRIPT ================= -->
{% if results %}
<script>
    const candidateNames = {{ results | map(attribute='leader_name') | list | tojson }};
    const voteCounts = {{ results | map(attribute='votes') | list | tojson }};

    const votes = voteCounts.map(v => parseInt(v));
    const sortedVotes = [...votes].sort((a, b) => b - a);

    function getRoyalBlueShade(position, total) {

        const ratio = position / (total - 1 || 1);

        // DARK → LIGHT Royal Blue scale
        const r = Math.round(10 + (100 * ratio));   // 10 → 110
        const g = Math.round(40 + (140 * ratio));   // 40 → 180
        const b = Math.round(120 + (120 * ratio));  // 120 → 240

        return `rgba(${r}, ${g}, ${b}, 0.95)`;
    }

    const totalCandidates = votes.length;

    const backgroundColors = votes.map(v => {
        const position = sortedVotes.indexOf(v);
        return getRoyalBlueShade(position, totalCandidates);
    });

    const borderColors = backgroundColors.map(c =>
        c.replace("0.95", "1")
    );

    const ctx = document.getElementById('voteChart');

    new Chart(ctx, {
        type: 'bar',
        data: {
            labels: candidateNames,
            datasets: [{
                label: 'Votes',
                data: votes,
                backgroundColor: backgroundColors,
                borderColor: borderColors,
                borderWidth: 2,
                borderRadius: 14
            }]
        },
        options: {
            responsive: true,
            plugins: {
                legend: {
                    labels: {
                        font: { size: 14 }
                    }
                }
            },
            scales: {
                y: {
                    beginAtZero: true,
                    ticks: { precision: 0 }
                }
            },
            animation: {
                duration: 1400,
                easing: 'easeOutCubic'
            }
        }
    });
</script>


{% endif %}

</body>
</html>


Overwriting templates/results.html


In [58]:
%%writefile templates/officer_login.html
<!DOCTYPE html>
<html>
<head>
    <title>Officer Login</title>
    <link rel="stylesheet"
          href="{{ url_for('static', filename='css/style.css') }}">
</head>
<body>

<div class="container">

    <h1>👮 Officer Login</h1>

    <form method="POST">

        <input type="text" name="username"
               placeholder="Username" required><br><br>

        <input type="password" name="password"
               placeholder="Password" required><br><br>

        <button class="primary-btn">Login</button>

    </form>

    {% if error %}
    <div class="error-box">
        {{ error }}
    </div>
    {% endif %}

</div>

</body>
</html>


Overwriting templates/officer_login.html


In [59]:
%%writefile templates/officer_dashboard.html
<!DOCTYPE html>
<html>
<head>
    <title>Officer Dashboard</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='css/style.css') }}">

<style>
body {
    margin: 0;
    font-family: Arial, sans-serif;
    background: linear-gradient(to right, #0f2027, #203a43, #2c5364);
}

/* Main Card */
.dashboard-card {
    width: 92%;
    margin: 40px auto;
    background: #ffffff;
    padding: 30px;
    border-radius: 15px;
    box-shadow: 0 15px 40px rgba(0,0,0,0.35);
    color: black; /* ✅ Force black text */
}

h2 {
    text-align: center;
    color: #0b3d91;
    margin-bottom: 25px;
}

/* Summary Boxes */
.summary {
    display: flex;
    justify-content: space-around;
    margin-bottom: 30px;
}

.summary-box {
    padding: 18px 30px;
    border-radius: 12px;
    color: white;
    font-weight: bold;
    text-align: center;
    min-width: 160px;
    font-size: 16px;
}

.waiting-box { background: #ffc107; color: black; }
.approved-box { background: #28a745; }
.rejected-box { background: #dc3545; }

/* Table */
table {
    width: 100%;
    border-collapse: collapse;
    background: white;
}

th {
    background-color: #0b3d91;
    color: white;
    padding: 14px;
    font-size: 15px;
}

td {
    padding: 14px;
    text-align: center;
    border-bottom: 1px solid #ddd;
    color: black; /* ✅ FIXED */
    font-weight: 500;
}

tr:hover {
    background-color: #f2f7ff;
}

/* Status Badges */
.status-approved {
    color: white;
    background: #28a745;
    padding: 6px 12px;
    border-radius: 8px;
    font-size: 13px;
}

.status-waiting {
    color: black;
    background: #ffc107;
    padding: 6px 12px;
    border-radius: 8px;
    font-size: 13px;
}

.status-rejected {
    color: white;
    background: #dc3545;
    padding: 6px 12px;
    border-radius: 8px;
    font-size: 13px;
}

/* Buttons */
.btn-approve {
    background: #28a745;
    color: white;
    border: none;
    padding: 8px 16px;
    border-radius: 8px;
    cursor: pointer;
    font-weight: bold;
}

.btn-reject {
    background: #dc3545;
    color: white;
    border: none;
    padding: 8px 16px;
    border-radius: 8px;
    cursor: pointer;
    font-weight: bold;
    margin-left: 5px;
}

.btn-approve:hover,
.btn-reject:hover {
    opacity: 0.85;
}

/* Footer Info */
.auto-refresh-info {
    text-align: center;
    margin-top: 20px;
    font-size: 13px;
    color: #555;
}
</style>
</head>

<body>

<div class="dashboard-card">

    <h2>🛡 OTP Approval Dashboard</h2>

    <!-- Summary -->
    <div class="summary">
        <div class="summary-box waiting-box">
            Waiting<br>
            <span id="waitingCount">{{ waiting_count }}</span>
        </div>

        <div class="summary-box approved-box">
            Approved<br>
            <span id="approvedCount">{{ approved_count }}</span>
        </div>

        <div class="summary-box rejected-box">
            Rejected<br>
            <span id="rejectedCount">{{ rejected_count }}</span>
        </div>
    </div>

    <!-- Table -->
    <table>
        <thead>
            <tr>
                <th>Voter ID</th>
                <th>OTP</th>
                <th>Status</th>
                <th>Action</th>
            </tr>
        </thead>

        <tbody id="table-body">
        {% for voter in voters %}
        <tr>
            <td>{{ voter.voter_id }}</td>
            <td>{{ voter.otp }}</td>

            <td>
                {% if voter.status == "APPROVED" %}
                    <span class="status-approved">Approved</span>
                {% elif voter.status == "REJECTED" %}
                    <span class="status-rejected">Rejected</span>
                {% elif voter.status == "EXPIRED" %}
                    <span class="status-rejected">Expired</span>
                {% else %}
                    <span class="status-waiting">Waiting</span>
                {% endif %}
            </td>

            <td>
                {% if voter.status == "WAITING" %}
                    <a href="{{ url_for('approve_otp_route', voter_id=voter.voter_id) }}">
                        <button class="btn-approve">Approve</button>
                    </a>

                    <a href="{{ url_for('reject_otp_route', voter_id=voter.voter_id) }}">
                        <button class="btn-reject">Reject</button>
                    </a>
                {% else %}
                    <span style="color:#28a745; font-weight:bold;">✅ Done</span>
                {% endif %}
            </td>
        </tr>
        {% endfor %}
        </tbody>
    </table>

    <div class="auto-refresh-info">
        🔄 Live updates every 5 seconds
    </div>

</div>

<script>
function refreshDashboard() {
    fetch("/officer_dashboard_data")
        .then(response => response.json())
        .then(data => {

            document.getElementById("waitingCount").innerText = data.waiting;
            document.getElementById("approvedCount").innerText = data.approved;
            document.getElementById("rejectedCount").innerText = data.rejected;

            document.getElementById("table-body").innerHTML = data.table_html;
        });
}

setInterval(refreshDashboard, 5000);
</script>

</body>
</html>


Overwriting templates/officer_dashboard.html


In [60]:
%%writefile templates/officer_table_partial.html

{% for voter in voters %}
<tr>
    <!-- Voter Name -->
    <td style="color:black;">
        {{ voter.voter_id }}


       
    </td>
        

    <!-- OTP -->
    <td style="color:black;">
        
        {{ voter.otp }}

        
    </td>

    <!-- Status -->
    <td>
        {% if voter.status == "APPROVED" %}
            <span class="status-approved">Approved</span>

        {% elif voter.status == "REJECTED" %}
            <span class="status-rejected">Rejected</span>

        {% elif voter.status == "EXPIRED" %}
            <span class="status-rejected">Expired</span>

        {% else %}
            <span class="status-waiting">Waiting</span>
        {% endif %}
    </td>

    <!-- Action -->
    <td>
        {% if voter.status == "WAITING" %}

            <a href="{{ url_for('approve_otp_route', voter_id=voter.voter_id) }}">
                <button class="btn-approve">Approve</button>
            </a>

            <a href="{{ url_for('reject_otp_route', voter_id=voter.voter_id) }}">
                <button class="btn-reject">Reject</button>
            </a>

        {% else %}
            <span style="color:#28a745; font-weight:bold;">✅ Done</span>
        {% endif %}
    </td>
</tr>
{% endfor %}

{% if voters|length == 0 %}
<tr>
    <td colspan="4" style="text-align:center; padding:20px; color:#777;">
        No OTP requests available
    </td>
</tr>
{% endif %}


Overwriting templates/officer_table_partial.html


In [61]:

%%writefile app.py
from flask import Flask, render_template, request, redirect, url_for, session, jsonify
import os
import pandas as pd
import secrets
import phase2_face_liveness as phase2
from flask import request, jsonify, session
import phase1_identity
import phase2_face_liveness
import phase3_offline_otp
import phase4_voting
from phase5_results import get_results_data, verify_admin
from flask import Response
import json
import time

# ======================================================
# APP CONFIGURATION
# ======================================================

app = Flask(__name__)
app.secret_key = secrets.token_hex(32)
app.config["SESSION_PERMANENT"] = False


# ======================================================
# HELPER
# ======================================================

def reset_session():
    session.clear()


# ======================================================
# HOME
# ======================================================

@app.route("/")
def index():
    reset_session()
    return render_template("index.html")


# ======================================================
# PHASE 1 - IDENTITY
# ======================================================

@app.route("/identity", methods=["GET", "POST"])
def identity():

    # ==============================
    # HANDLE FORM SUBMISSION
    # ==============================
    if request.method == "POST":

        voter_id = request.form.get("voter_id", "").strip()

        # 🔹 Check empty input
        if not voter_id:
            return render_template(
                "identity.html",
                error="Please enter Voter ID."
            )

        # 🔹 Verify identity using Phase 1
        result = phase1_identity.verify_identity(voter_id)

        # 🔹 If voter ID invalid
        if not result.get("status"):
            return render_template(
                "identity.html",
                error=result.get("reason", "Invalid Voter ID.")
            )

        voter_data = result.get("data", {})

        # 🔥 BLOCK ALREADY VOTED USERS
        if voter_data.get("has_voted", "").strip().lower() == "yes":
            return render_template(
                "identity.html",
                error="❌ You have already voted. Duplicate voting is not allowed."
            )

        # 🔹 Reset old session safely
        reset_session()

        # 🔹 Store voter details in session
        session["identity_verified"] = True
        session["voter_id"] = voter_data.get("voter_id")
        session["face_id"] = voter_data.get("face_id")
        session["voter_name"] = voter_data.get("name")
        session["village"] = voter_data.get("village")
        session["voting_center"] = voter_data.get("voting_center")

        return redirect(url_for("details"))

    # ==============================
    # LOAD PAGE (GET REQUEST)
    # ==============================
    return render_template("identity.html")

# ======================================================
# DETAILS CONFIRMATION
# ======================================================

@app.route("/details", methods=["GET", "POST"])
def details():

    if not session.get("identity_verified"):
        return redirect(url_for("identity"))

    face_id = session.get("face_id")
    folder_path = os.path.join("static", "images", "voters", face_id)

    image_file = "img1.jpg"

    if os.path.exists(folder_path):
        files = os.listdir(folder_path)
        if "img.jpeg" in files:
            image_file = "img.jpeg"
        elif "img1.jpg" in files:
            image_file = "img1.jpg"

    image_path = f"images/voters/{face_id}/{image_file}"

    if request.method == "POST":
        session["details_confirmed"] = True
        return redirect(url_for("face_camera"))

    return render_template("details.html", image_path=image_path)


# ======================================================
# PHASE 2 - FACE LIVENESS
# ======================================================
# ======================================================
# FACE INSTRUCTION
# ======================================================

@app.route("/face")
def face():

    if not session.get("details_confirmed"):
        return redirect(url_for("identity"))

    return render_template("face.html")


# ======================================================
# FACE CAMERA
# ======================================================

@app.route("/face_camera")
def face_camera():

    if not session.get("details_confirmed"):
        return redirect(url_for("identity"))

    return render_template("face_camera.html")


# ======================================================
# FACE VERIFICATION API
# ======================================================

@app.route("/api/verify_face", methods=["POST"])
def api_verify_face():

    if not session.get("details_confirmed"):
        return jsonify({"status": False, "message": "Session expired"})

    # ✅ Initialize liveness once per session
    if "liveness_initialized" not in session:
        phase2_face_liveness.reset_liveness()
        session["liveness_initialized"] = True

    data = request.get_json()
    image_data = data.get("image")

    if not image_data:
        return jsonify({"status": False, "message": "No image received"})

    face_id = session.get("face_id")

    result = phase2_face_liveness.process_frame(face_id, image_data)

    if result["status"]:
        session["face_verified"] = True
        session.pop("liveness_initialized", None)
        return jsonify({"status": True})

    return jsonify({
        "status": False,
        "message": result.get("message", "Face verification failed")
    })

# ======================================================
# PHASE 3 - OFFICER CONTROLLED OTP
# ======================================================

@app.route("/otp")
def otp():

    if not session.get("face_verified"):
        return redirect(url_for("identity"))

    voter_id = session.get("voter_id")

    if not voter_id:
        return redirect(url_for("identity"))

    if not session.get("otp_session_started"):
        phase3_offline_otp.start_otp_session(session, voter_id)

    return render_template("otp.html")

# ======================================================
# CHECK OTP STATUS (AJAX POLLING)
# ======================================================

@app.route("/check_otp_status")
def check_otp_status():

    voter_id = session.get("voter_id")

    if not voter_id:
        return jsonify({"status": "NO_SESSION"})

    result = phase3_offline_otp.check_otp_status(voter_id)

    if result["status"] == "APPROVED":
        session["otp_verified"] = True

    return jsonify(result)


# ======================================================
# REAL-TIME OTP STREAM (SSE)
# ======================================================


@app.route("/otp_stream")
def otp_stream():

    voter_id = session.get("voter_id")

    if not voter_id:
        return Response(
            "data: {\"status\":\"NO_SESSION\"}\n\n",
            mimetype="text/event-stream",
            headers={
                "Cache-Control": "no-cache",
                "X-Accel-Buffering": "no"
            }
        )

    def event_stream():

        last_status = None

        while True:
            result = phase3_offline_otp.check_otp_status(voter_id)
            current_status = result["status"]

            if current_status != last_status:
                yield f"data: {json.dumps(result)}\n\n"
                last_status = current_status

            time.sleep(0.3)

    return Response(
        event_stream(),
        mimetype="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",
            "Connection": "keep-alive"
        }
    )
# ======================================================
# PHASE 4 - VOTING
# ======================================================
@app.route("/voting", methods=["GET", "POST"])
def voting():

    voter_id = session.get("voter_id")

    if not voter_id:
        return redirect(url_for("identity"))

    otp_status = phase3_offline_otp.check_otp_status(voter_id)["status"]

    if otp_status != "APPROVED":
        return redirect(url_for("identity"))

    candidates = phase4_voting.load_candidates()

    if request.method == "POST":

        candidate_id = request.form.get("selected_candidate")

        result = phase4_voting.cast_vote(voter_id, candidate_id)

        if result["status"]:
            reset_session()
            return redirect(url_for("vote_success"))

        error_map = {
            "ALREADY_VOTED": "You have already voted.",
            "INVALID_CANDIDATE": "Invalid candidate selected.",
            "INVALID_VOTER": "Invalid voter."
        }

        return render_template("voting.html",
                               candidates=candidates,
                               error=error_map.get(result.get("reason"),
                                                   "System error occurred."))

    return render_template("voting.html",
                           candidates=candidates)
# ======================================================
# VOTE SUCCESS
# ======================================================

@app.route("/vote_success")
def vote_success():
    return render_template("vote_success.html")


# ======================================================
# OFFICER LOGIN
# ======================================================

@app.route("/officer_login", methods=["GET", "POST"])
def officer_login():

    if request.method == "POST":

        username = request.form.get("username")
        password = request.form.get("password")

        if username == "officer" and password == "1234":
            
            session["officer"] = True
            return redirect(url_for("officer_dashboard"))

        return render_template("officer_login.html",
                               error="Invalid Officer Credentials")

    return render_template("officer_login.html")


# ======================================================
# OFFICER DASHBOARD
# ======================================================

@app.route("/officer_dashboard")
def officer_dashboard():

    if not session.get("officer"):
        return redirect(url_for("officer_login"))

    otp_file = "data/otp_sessions.csv"
    voters = []

    if os.path.exists(otp_file):
        df = pd.read_csv(otp_file)
        voters = df.to_dict(orient="records")

    waiting = sum(1 for v in voters if v["status"] == "WAITING")
    approved = sum(1 for v in voters if v["status"] == "APPROVED")
    rejected = sum(1 for v in voters if v["status"] == "REJECTED")

    return render_template(
        "officer_dashboard.html",
        voters=voters,
        waiting_count=waiting,
        approved_count=approved,
        rejected_count=rejected
    )


# ======================================================
# APPROVE / REJECT
# ======================================================

@app.route("/approve_otp/<voter_id>")
def approve_otp_route(voter_id):

    if not session.get("officer"):
        return redirect(url_for("officer_login"))

    phase3_offline_otp.approve_otp(voter_id)
    return redirect(url_for("officer_dashboard"))


@app.route("/reject_otp/<voter_id>")
def reject_otp_route(voter_id):

    if not session.get("officer"):
        return redirect(url_for("officer_login"))

    phase3_offline_otp.reject_otp(voter_id)
    return redirect(url_for("officer_dashboard"))


# ======================================================
# ADMIN LOGIN
# ======================================================

@app.route("/admin", methods=["GET", "POST"])
def admin():

    if request.method == "POST":

        username = request.form.get("username")
        password = request.form.get("password")

        if verify_admin(username, password):
            
            session["admin"] = True
            return redirect(url_for("results"))

        return render_template("admin_login.html",
                               error="Invalid credentials")

    return render_template("admin_login.html")


# ======================================================
# RESULTS
# ======================================================

@app.route("/results")
def results():

    if not session.get("admin"):
        return redirect(url_for("admin"))

    results_data = get_results_data()

    if not results_data:
        return render_template("results.html",
                               results=None,
                               winner="No Data",
                               total=0,
                               cast=0,
                               turnout=0)

    results_list = results_data["data"]
    votes_cast = sum(r["votes"] for r in results_list)

    winner = "Tie Between Multiple Candidates" \
        if results_data["tie"] \
        else results_data["winners"][0].get("leader_name", "Unknown")

    voter_file = "data/voter_details.csv"
    total_voters = len(pd.read_csv(voter_file)) if os.path.exists(voter_file) else 0

    turnout = round((votes_cast / total_voters) * 100, 2) if total_voters else 0

    return render_template("results.html",
                           results=results_list,
                           winner=winner,
                           total=total_voters,
                           cast=votes_cast,
                           turnout=turnout)


# ======================================================
# LOGOUT
# ======================================================

@app.route("/logout")
def logout():
    reset_session()
    return redirect(url_for("index"))


# ======================================================
# RUN (LAN / HOTSPOT ENABLED)
# ======================================================

if __name__ == "__main__":
    
    app.run(host="0.0.0.0", port=5000, debug=True, use_reloader=False, threaded=True)

Overwriting app.py
